In [ ]:
from pathlib import Path

path = Path("data/osm/california-260203.osm.pbf/").resolve()

print(path)

### Runtimes
Sparse index + filters: 4m 27.3s  
Filters: 3m 53.8s  
Handlerv1 (355664 cells): 5m 5.0s  
Handlerv2 (355581 cells): 10m 35.6s  
Handlerv3 (336732 cells): 10m 33.4s
Handlerv3 run 2(355581 cells): 8m 20.5s  

In [ ]:
from classes.scenic_handler_v1 import ScenicHandler as ScenicHandlerV1
import json
import time
from osmium.filter import KeyFilter

start = time.time()

RESOLUTION = 8

water_data = json.load(open("constants/natural_water_tags.json"))
vegetation_data = json.load(open("constants/natural_vegetation_tags.json"))
geological_data = json.load(open("constants/natural_geological_tags.json"))
waterway_data = json.load(open("constants/waterway_tags.json"))

handler = ScenicHandlerV1(
    water_data, vegetation_data, geological_data, waterway_data, resolution=RESOLUTION
)
handler.apply_file(
    path,
    locations=True,
    idx="sparse_file_array",
    filters=[
        KeyFilter("natural", "landcover", "waterway", "tourism", "landuse", "leisure")
    ],
)

elapsed = time.time() - start
print(f"Took {elapsed:.2f}s")

print(f"Parsed {len(handler.cells)} H3 cells")

### Runtime: 

In [ ]:
import json

with open("data/output/scenic_cells.json", "w") as f:
    json.dump(handler.cells, f)

print(f"Saved {len(handler.cells)} H3 cells")

In [ ]:
import pandas as pd
import json
import os
import numpy as np

print(os.getcwd())

In [ ]:
# with open("data/output/scenic_cells_v1.json", "r") as f:
#     cells = json.load(f)

cells = handler.cells

In [ ]:
# Convert to DataFrame for easy scoring
df = pd.DataFrame.from_dict(cells, orient="index")
df.index.name = "h3_cell"
df.reset_index(inplace=True)

# Scenic score formula
df["diversity"] = (df[["water", "landcover", "relief", "recreation", "viewpoint"]] > 0).sum(axis=1)

feature_cols = ["water", "landcover", "relief", "recreation", "viewpoint", "urban"]
df[feature_cols] = df[feature_cols].apply(np.log1p)

df["raw_score"] = (
    4 * np.sqrt(df["water"])
    + 3 * np.sqrt(df["relief"])
    + 2 * np.sqrt(df["landcover"])
    + 2 * np.sqrt(df["recreation"])
    + 1.5 * df["viewpoint"]
    + 2 * np.log1p(df["diversity"])
    - 1 * np.sqrt(df["urban"])
)

# Normalize to 0–100
df["score"] = df["raw_score"].rank(pct=True) * 100

df_ranked = df.sort_values("score", ascending=False)
print(df_ranked[["h3_cell", "score"]].head(200))

In [ ]:
from datetime import datetime
from pathlib import Path

output_dir = Path("data/output") / datetime.now().strftime("%Y%m%d")
output_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%H%M%S")

df_ranked.to_json(output_dir / f"scenic_scores_{ts}.json", orient="records")
df_ranked.to_csv(output_dir / f"scenic_scores_{ts}.csv", index=False)

In [ ]:
# Detect skew
print("Raw score:")
print(df["raw_score"].describe())
print()
print(df["raw_score"].quantile([0.1, 0.33, 0.5, 0.75, 0.9, 0.95, 0.99]))

In [ ]:
print("\nFinal Score")
print(df["score"].describe())
print()
print(df["score"].quantile([0.1, 0.33, 0.5, 0.75, 0.9, 0.95, 0.99]))

In [ ]:
# find mammoth lakes cells
mammoth_lat, mammoth_lng = 37.6488, -118.9718
import h3

mammoth_cell = h3.latlng_to_cell(mammoth_lat, mammoth_lng, RESOLUTION)
print(df[df["h3_cell"] == mammoth_cell])